# Forward Pass — How Data Flows Through a Model

The forward pass is the journey of input data through every layer until it produces an output.
Understanding it at each step is critical — shape mismatches and bugs always happen here.

**What we cover:**
- Tracing shapes through each layer
- What each layer actually computes
- Common shape bugs and how to debug them
- How `model(x)` calls `forward(x)` under the hood

In [ ]:
import torch
import torch.nn as nn

# --- Trace shapes step by step ---
# Build a model that prints shape at each step

class TracedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        print(f'Input:      {x.shape}')

        x = self.fc1(x)
        print(f'After fc1:  {x.shape}')   # [batch, 256]

        x = self.relu(x)
        print(f'After relu: {x.shape}')   # same shape — activation doesn't change shape

        x = self.fc2(x)
        print(f'After fc2:  {x.shape}')   # [batch, 128]

        x = self.fc3(x)
        print(f'After fc3:  {x.shape}')   # [batch, 10]

        return x

model = TracedMLP()
x = torch.randn(4, 784)   # batch of 4, each is a flattened 28×28 image
out = model(x)

## What nn.Linear actually computes

`nn.Linear(in, out)` stores a weight matrix W of shape `[out, in]` and bias `[out]`.

When you call `layer(x)`: computes `x @ W.T + b`

Note `W.T` — PyTorch stores weights transposed for efficiency.

In [ ]:
layer = nn.Linear(4, 3)   # in=4, out=3

print('weight shape:', layer.weight.shape)   # [3, 4] — stored as [out, in]
print('bias shape:  ', layer.bias.shape)     # [3]

x = torch.randn(2, 4)   # batch=2, features=4

# What PyTorch does:
auto_out   = layer(x)
manual_out = x @ layer.weight.T + layer.bias   # manually doing the same thing

print('\nAuto output:  ', auto_out)
print('Manual output:', manual_out)
print('Same?         ', torch.allclose(auto_out, manual_out))

## Common shape bugs — how to debug

Shape mismatches are the #1 error in PyTorch. Always check shapes when debugging.

In [ ]:
# BUG 1: forgetting to flatten before a Linear layer
model = nn.Sequential(
    nn.Linear(784, 128),   # expects [batch, 784]
    nn.ReLU(),
    nn.Linear(128, 10)
)

# Input as image (not flattened)
img = torch.randn(4, 1, 28, 28)   # [batch, channels, H, W]
try:
    out = model(img)
except RuntimeError as e:
    print('BUG 1 - forgot Flatten:', e)

# Fix: add nn.Flatten() at the start
model_fixed = nn.Sequential(
    nn.Flatten(),           # [4, 1, 28, 28] → [4, 784]
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
out = model_fixed(img)
print('Fixed output shape:', out.shape)

In [ ]:
# BUG 2: missing squeeze/unsqueeze — dimension mismatch in loss
model  = nn.Linear(4, 1)   # outputs [batch, 1]
target = torch.randn(8)    # shape [8] — no extra dim
x      = torch.randn(8, 4)

pred   = model(x)          # shape [8, 1]
print('pred shape:  ', pred.shape)    # [8, 1]
print('target shape:', target.shape)  # [8]

loss_fn = nn.MSELoss()
try:
    loss = loss_fn(pred, target)
except Exception as e:
    print('BUG 2 - shape mismatch in loss')

# Fix: squeeze the output
loss = loss_fn(pred.squeeze(), target)
print('Fixed — loss:', loss.item())

In [ ]:
# DEBUGGING TIP: use hooks to print shapes during forward pass
# without modifying the model

model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4)
)

handles = []
for name, layer in model.named_modules():
    if isinstance(layer, nn.Linear):
        def hook(module, input, output, name=name):
            print(f'{name}: input {input[0].shape} → output {output.shape}')
        handles.append(layer.register_forward_hook(hook))

x = torch.randn(3, 8)
out = model(x)

# Clean up hooks
for h in handles:
    h.remove()

## How model(x) → forward(x) works

When you write `model(x)`, you're not calling `forward()` directly.
You're calling `nn.Module.__call__()`, which:
1. Runs registered forward hooks
2. Calls `forward(x)`
3. Runs registered backward hooks

This is why you should NEVER call `model.forward(x)` directly — you'd bypass hooks.

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        return self.fc(x)

model = MyModel()
x = torch.randn(2, 4)

# Correct — goes through __call__ (hooks, etc.)
out1 = model(x)

# Wrong — bypasses __call__ machinery
out2 = model.forward(x)

print('Same output:', torch.allclose(out1, out2))   # True, but model(x) is the right way